In [1]:
import joblib
import numpy as np
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [2]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-22-08_54_45_PM'

In [3]:
oof_experiment_config = {
     "experiment": {
        "model": "xgboost",
        "type": "baseline",
        "train_feature_groups": ["baseline/train.parq", "features/train.parq", "target_encode/train.parq"],
        "test_feature_groups": ["baseline/test.parq", "features/test.parq", "target_encode/test.parq"],
        "description": "xgb+clf+baseline+features+TE with opt params and dart booster",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "features": {
        "class_sample_weight": False
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": "dart",
        "rate_drop": 0.8,
        "n_estimators": 5000,
        "learning_rate": 0.03,
        "max_depth": 7,
        "min_child_weight": 1.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "max_bin": 256,
        "tree_method": "hist",
        "enable_categorical": True,
        "device": "cuda",
        "sampling_method": "uniform",
        "early_stopping_rounds": 200,
        "n_jobs": -1,
        "random_state": 4
    },
    
    "fit_params": {
    }
}
oof_experiment_config["experiment"]["id"] = f"{dt_str}_{oof_experiment_config["experiment"]["model"]}"
experiment_config = oof_experiment_config.copy()

In [4]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [5]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [6]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
X_dfs, X_test_dfs = [], []

for fg in oof_experiment_config["experiment"]["train_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_dfs.append(df)

for fg in oof_experiment_config["experiment"]["test_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_test_dfs.append(df)

X = pd.concat(X_dfs, axis=1)
X_test = pd.concat(X_test_dfs, axis=1)
y = raw_train_df[target_column]

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 40 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              64714

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 40 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              64714

In [8]:
def make_model(config):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor), # TODO - fix missing col issue in catboost
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
        "tabpfn": (TabPFNClassifier, TabPFNRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params)

In [9]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp", "tabpfn"]
    fit_params = config["fit_params"]
    
    if "features" in config and "class_sample_weight" in config["features"] and config["features"]["class_sample_weight"]:
        train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    else:
        train_sample_weight = None

    if name in no_eval_models:
        model.fit(X_train, y_train, sample_weight=train_sample_weight)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid, sample_weight=train_sample_weight)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight)

In [10]:
def predict(config, model, X_feat):
    task = config["experiment"]["target"]
    if task == "classification":
        return model.predict_proba(X_feat)[:, 1]
    else:
        return model.predict(X_feat)

In [11]:
cv_config = experiment_config["cv"]
kf = StratifiedKFold(n_splits=cv_config["n_splits"], random_state=cv_config["random_state"], shuffle=cv_config["shuffle"])

y_cv = pd.Series(index=y.index, dtype=float, name=target_column)
fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = make_model(oof_experiment_config)
    oof_fit(oof_experiment_config, model, X_train, y_train, X_valid, y_valid)

    y_pred = predict(oof_experiment_config, model, X_valid)
    y_pred_clip = np.clip(y_pred, 0.0, 1.0)
    y_cv.iloc[valid_index] = y_pred_clip

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = pd.concat([raw_train_id, y_cv], axis=1)

[0]	validation_0-auc:0.94420
[1]	validation_0-auc:0.94689
[2]	validation_0-auc:0.94916
[3]	validation_0-auc:0.95000
[4]	validation_0-auc:0.95101
[5]	validation_0-auc:0.95168
[6]	validation_0-auc:0.95202
[7]	validation_0-auc:0.95279
[8]	validation_0-auc:0.95282
[9]	validation_0-auc:0.95285
[10]	validation_0-auc:0.95294
[11]	validation_0-auc:0.95298
[12]	validation_0-auc:0.95300
[13]	validation_0-auc:0.95310
[14]	validation_0-auc:0.95314
[15]	validation_0-auc:0.95318
[16]	validation_0-auc:0.95317
[17]	validation_0-auc:0.95316
[18]	validation_0-auc:0.95318
[19]	validation_0-auc:0.95320
[20]	validation_0-auc:0.95324
[21]	validation_0-auc:0.95325
[22]	validation_0-auc:0.95326
[23]	validation_0-auc:0.95328
[24]	validation_0-auc:0.95329
[25]	validation_0-auc:0.95332
[26]	validation_0-auc:0.95333
[27]	validation_0-auc:0.95333
[28]	validation_0-auc:0.95334
[29]	validation_0-auc:0.95335
[30]	validation_0-auc:0.95334
[31]	validation_0-auc:0.95340
[32]	validation_0-auc:0.95349
[33]	validation_0-au

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [04:33:46] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[0]	validation_0-auc:0.94335
[1]	validation_0-auc:0.94598
[2]	validation_0-auc:0.94816
[3]	validation_0-auc:0.94907
[4]	validation_0-auc:0.95006
[5]	validation_0-auc:0.95085
[6]	validation_0-auc:0.95132
[7]	validation_0-auc:0.95205
[8]	validation_0-auc:0.95210
[9]	validation_0-auc:0.95213
[10]	validation_0-auc:0.95220
[11]	validation_0-auc:0.95224
[12]	validation_0-auc:0.95228
[13]	validation_0-auc:0.95238
[14]	validation_0-auc:0.95242
[15]	validation_0-auc:0.95245
[16]	validation_0-auc:0.95244
[17]	validation_0-auc:0.95242
[18]	validation_0-auc:0.95244
[19]	validation_0-auc:0.95246
[20]	validation_0-auc:0.95248
[21]	validation_0-auc:0.95248
[22]	validation_0-auc:0.95249
[23]	validation_0-auc:0.95251
[24]	validation_0-auc:0.95251
[25]	validation_0-auc:0.95252
[26]	validation_0-auc:0.95253
[27]	validation_0-auc:0.95253
[28]	validation_0-auc:0.95254
[29]	validation_0-auc:0.95254
[30]	validation_0-auc:0.95254
[31]	validation_0-auc:0.95261
[32]	validation_0-auc:0.95269
[33]	validation_0-au

In [12]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9528902782393703


In [13]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "train_feature_groups": f"{experiment_config["experiment"]["train_feature_groups"]}",
    "cv": {
        "strategy": f"{experiment_config["cv"]["strategy"]}",
        "n_splits": experiment_config["cv"]["n_splits"],
        "random_state": experiment_config["cv"]["random_state"],
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [14]:
params = experiment_config["params"]
tree_param_keys = ["n_estimators", "max_iter"]

best_iter = None
if hasattr(model, "best_iteration_"):
    best_iter = model.best_iteration_
elif hasattr(model, "best_iteration"):
    best_iter = model.best_iteration
elif hasattr(model, "n_iter_"):
    n_iter_val = model.n_iter_
    
    if isinstance(n_iter_val, np.ndarray):
        if n_iter_val.size == 1:
            best_iter = n_iter_val.item()
        else:
            best_iter = int(n_iter_val.max())
    else:
        best_iter = int(n_iter_val)

if best_iter is not None:
    for key in tree_param_keys:
        if key in params:
            params[key] = best_iter

training_only_params = [
    "early_stopping_rounds",
    "early_stopping",
    "n_iter_no_change",
    "validation_fraction",
    "eval_set",
]

for key in training_only_params:
    if key in params:
        del params[key]

In [15]:
if experiment_config["features"]["class_sample_weight"]:
    y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)
else:
    y_sample_weight = None

model = make_model(experiment_config)
model.fit(X, y, sample_weight=y_sample_weight)

y_pred = predict(experiment_config, model, X_test)
y_pred_clip = np.clip(y_pred, 0.0, 1.0)
ss[target_column] = y_pred_clip
ss

,id,addicted_label
0,691369,0.784820
1,691370,0.760553
2,691371,0.698822
3,691372,0.781088
4,691373,0.781871
...,...,...
296297,987666,0.785331
296298,987667,0.740971
296299,987668,0.620722
296300,987669,0.614714


In [16]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "oof_config.json", "w") as f:
    json.dump(oof_experiment_config, f, indent=4)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

y_pred_df.to_csv(experiment_path / "oof.csv", index=False)
joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")
ss.to_csv(experiment_path / f"{experiment_config["experiment"]["model"]}_submission.csv", index=False)